In [ ]:
import pandas as pd

all_pr = pd.read_parquet(
    "hf://datasets/hao-li/AIDev/all_pull_request.parquet",
    columns=["id", "agent", "title"]
)
print(len(all_pr), "PRs total")
title = all_pr["title"].fillna("").str.lower()
strict = title.str.match(r"^\s*refactor\b\s*(\([^)]*\))?\s*!?\s*:")
loose = title.str.contains(r"\brefactor", regex=True)
print("Refactoring PRs (strict prefix):", strict.sum())
print("Refactoring PRs (keyword anywhere):", loose.sum())
refactor_all = all_pr[loose]
print(refactor_all["agent"].value_counts())

2743854 PRs total
Refactoring PRs (strict prefix): 30894
Refactoring PRs (keyword anywhere): 94508
agent
OpenAI_Codex    76305
Copilot          7530
Google_Jules     4510
Cursor           4077
Devin            1121
Claude_Code       965
Name: count, dtype: int64


In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.dataset as ds
from huggingface_hub import HfFileSystem

URL = "datasets/hao-li/AIDev/all_pull_request.parquet"
fs = HfFileSystem()
all_pr = pd.read_parquet("hf://" + URL, columns=["id", "agent", "title"])
mask = all_pr["title"].fillna("").str.contains(r"\brefactor", case=False, regex=True)
refactor_ids = all_pr.loc[mask, "id"].tolist()
print(len(refactor_ids), "refactoring PRs")
dataset = ds.dataset(URL, filesystem=fs, format="parquet")
refactor_df = dataset.to_table(
    columns=["id", "agent", "title", "body", "html_url"],
    filter=ds.field("id").isin(refactor_ids),
).to_pandas()

94508 refactoring PRs


In [ ]:
patterns = {
    "extract_method": r"\bextract(?:ed|s|ing)?\b[\w\s`'\"-]{0,40}\b(?:method|function|helper)s?\b",
    "move_method":    r"\bmov(?:e|ed|es|ing)\b[\w\s`'\"-]{0,40}\b(?:method|function)s?\b",
    "extract_class":  r"\bextract(?:ed|s|ing)?\b[\w\s`'\"-]{0,40}\b(?:class|classes|module|component|service|interface)\b",
    "extract_field":  r"\bextract(?:ed|s|ing)?\b[\w\s`'\"-]{0,40}\b(?:field|attribute|property|properties|constant|variable)s?\b",
    "move_field":     r"\bmov(?:e|ed|es|ing)\b[\w\s`'\"-]{0,40}\b(?:field|attribute|property|properties|constant|variable)s?\b",
}
text = (refactor_df["title"].fillna("") + " " + refactor_df["body"].fillna("")).str.lower()
for name, pat in patterns.items():
    refactor_df[name] = text.str.contains(pat, regex=True)
specific = list(patterns)
refactor_df = refactor_df[refactor_df[specific].any(axis=1)].copy()
print(refactor_df[specific].sum().sort_values(ascending=False))
print("Unique PRs:", len(refactor_df))

extract_method    2197
extract_class      909
move_method        423
extract_field      368
move_field         242
dtype: int64
Unique PRs: 3829
